# 5. Forecasting next month's SST anomaly

**What it does**
- Trains on 2001–2017 and tests on January 2018 – December 2022 (58 months with good coverage).
- Compares persistence, linear regression on lagged SST and season, the same regression plus lagged ICEN, and XGBoost.
- Reports MAE and RMSE overall and by ENSO phase. The phase labels come from notebook 1 and are only used to group the errors.

**Inputs**
- `data/processed/sst_forecast_data.csv` from notebook 3
- `data/processed/icen_monthly_pm04.csv` from notebook 1

**Outputs**
- Error tables (printed). Result: regression and XGBoost beat persistence by about 10% (0.195 vs 0.216 °C); ICEN adds almost nothing.

Run this notebook from the `notebooks/` folder, top to bottom.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Coastal El Niño / La Niña rule (same as notebook 1):
# El Niño when ICEN >= +0.4 °C, La Niña when ICEN <= -0.4 °C, for at least 3 consecutive months.
ENSO_WARM, ENSO_COLD, ENSO_MIN_MONTHS = 0.4, -0.4, 3

# Relabelled ICEN written by 1_enso_episodes.ipynb
ICEN_CSV = Path("../data/processed/icen_monthly_pm04.csv")
assert ICEN_CSV.exists(), "Run 1_enso_episodes.ipynb first"

## Baseline models: one-month SST anomaly forecast

Reads `sst_forecast_data.csv` from notebook 3.

### Persistence

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

sst = pd.read_csv("../data/processed/sst_forecast_data.csv")
icen = pd.read_csv(ICEN_CSV).rename(
    columns={"Year": "year", "Month": "month"}
)

data = sst.merge(
    icen[["year", "month", "ENSO", "Episode"]],
    on=["year", "month"],
    how="left",
    validate="one_to_one",
)

data["date"] = pd.to_datetime(data[["year", "month"]].assign(day=1))
data = data.set_index("date").sort_index().asfreq("MS")

# Keep gaps so the forecast always uses the previous calendar month.
data["actual"] = data["sst_anom"].where(data["low_coverage"].eq(False))
data["persistence"] = data["actual"].shift(1)

# Use the target month's ICEN label to group the errors after forecasting.
data["phase"] = data["Episode"].where(
    data["Episode"].isin(["El Niño", "La Niña"])
)
data.loc[data["ENSO"].eq("Neutral"), "phase"] = "Neutral"

test = data.loc["2018-01":"2022-12"].dropna(
    subset=["actual", "persistence"]
).copy()
test["absolute_error"] = (test["actual"] - test["persistence"]).abs()

print(f"Test months: {len(test)}")
print(f"Overall persistence MAE: {test['absolute_error'].mean():.3f} °C")

summary = (
    test.groupby("phase")["absolute_error"]
    .agg(Months="size", MAE="mean")
    .reindex(["El Niño", "La Niña", "Neutral"])
)
print("\nPersistence by phase (MAE in °C)")
print(summary.round(3).to_string())

Test months: 53
Overall persistence MAE: 0.239 °C

Persistence by phase (MAE in °C)
         Months    MAE
phase                 
El Niño       4  0.548
La Niña      34  0.213
Neutral      15  0.216


### Linear regression on lagged SST and season

In [3]:
# For a forecast of March, lag 1 is February and lag 2 is January.
data["sst_lag1"] = data["persistence"]
data["sst_lag2"] = data["actual"].shift(2)

# Encode the target month so December and January are close together.
angle = 2 * np.pi * (data.index.month - 1) / 12
data["month_sin"] = np.sin(angle)
data["month_cos"] = np.cos(angle)

features = ["sst_lag1", "sst_lag2", "month_sin", "month_cos"]
required = ["actual", *features]

train = data.loc[:"2017-12"].dropna(subset=required)
test_common = data.loc["2018-01":"2022-12"].dropna(subset=required).copy()

model = LinearRegression()
model.fit(train[features], train["actual"])
test_common["regression"] = model.predict(test_common[features])

print(f"\nTraining months: {len(train)}")
print(f"Shared test months: {len(test_common)}")
for name in ["persistence", "regression"]:
    mae = (test_common["actual"] - test_common[name]).abs().mean()
    print(f"{name.title()} MAE: {mae:.3f} °C")


Training months: 157
Shared test months: 52
Persistence MAE: 0.239 °C
Regression MAE: 0.217 °C


### Regression with lagged ICEN

In [4]:
data = data.join(icen.set_index(["year", "month"])["ICEN"], on=["year", "month"])

# ICEN is a centred 3-month index. Allow an extra month for publication.
# For a March forecast, this uses the ICEN value labelled December.
data["icen_lag3"] = data["ICEN"].shift(3)

features_with_icen = [*features, "icen_lag3"]
required = ["actual", *features_with_icen]

train_apped = data.loc[:"2017-12"].dropna(subset=required)
test_apped = data.loc["2018-01":"2022-12"].dropna(subset=required).copy()

print(f"\nTest months shared by all models: {len(test_apped)}")
mae = (test_apped["actual"] - test_apped["persistence"]).abs().mean()
print(f"Persistence MAE: {mae:.3f} °C")

for label, inputs in [
    ("Regression without ICEN", features),
    ("Regression with ICEN", features_with_icen),
]:
    model = LinearRegression().fit(train_apped[inputs], train_apped["actual"])
    prediction = model.predict(test_apped[inputs])
    mae = (test_apped["actual"] - prediction).abs().mean()
    print(f"{label} MAE: {mae:.3f} °C")


Test months shared by all models: 52
Persistence MAE: 0.239 °C
Regression without ICEN MAE: 0.217 °C
Regression with ICEN MAE: 0.217 °C


### XGBoost, and errors by ENSO phase

In [5]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42,
)

xgb.fit(train_apped[features_with_icen], train_apped["actual"])
test_apped["xgboost"] = xgb.predict(test_apped[features_with_icen])

mae = (test_apped["actual"] - test_apped["xgboost"]).abs().mean()
print(f"XGBoost with ICEN MAE: {mae:.3f} °C")

test_apped["regression_with_icen"] = model.predict(
    test_apped[features_with_icen]
)

predictions = test_apped[
    ["persistence", "regression_with_icen", "xgboost"]
].rename(columns={"regression_with_icen": "regression"})

errors = predictions.sub(test_apped["actual"], axis=0)

overall = pd.DataFrame({
    "MAE (°C)": errors.abs().mean(),
    "RMSE (°C)": errors.pow(2).mean().pow(0.5),
})

print(f"\nOverall one-month forecast errors ({len(errors)} test months)")
print(overall.round(3).to_string())

for phase in ["El Niño", "La Niña", "Neutral"]:
    phase_errors = errors.loc[test_apped["phase"].eq(phase)]

    summary = pd.DataFrame({
        "MAE (°C)": phase_errors.abs().mean(),
        "RMSE (°C)": phase_errors.pow(2).mean().pow(0.5),
    })

    print(f"\n{phase} ({len(phase_errors)} test months)")
    print(summary.round(3).to_string())

XGBoost with ICEN MAE: 0.224 °C

Overall one-month forecast errors (52 test months)
             MAE (°C)  RMSE (°C)
persistence     0.239      0.300
regression      0.217      0.279
xgboost         0.224      0.281

El Niño (4 test months)
             MAE (°C)  RMSE (°C)
persistence     0.548      0.621
regression      0.494      0.559
xgboost         0.435      0.511

La Niña (33 test months)
             MAE (°C)  RMSE (°C)
persistence     0.212      0.257
regression      0.193      0.241
xgboost         0.204      0.254

Neutral (15 test months)
             MAE (°C)  RMSE (°C)
persistence     0.216      0.253
regression      0.195      0.241
xgboost         0.212      0.250
